In [3]:
import sys
sys.path.append('..')
import pandas as pd
from src.stock_data_loader import load_stock, fix_dtypes, handle_missing

### Load and clean news

In [ ]:
df_news = pd.read_csv('../data/raw/raw_analyst_ratings.csv', index_col=0)

df_news['date'] = pd.to_datetime(df_news['date'].str.replace(r'-\d{2}:\d{2}$', '', regex=True))

# Extract just the date (remove time)
df_news['news_date'] = df_news['date'].dt.date

print(f"Loaded {len(df_news):,} news articles")
print(f"Date range: {df_news['news_date'].min()} to {df_news['news_date'].max()}")

Loaded 1,407,328 news articles
Date range: 2009-02-14 to 2020-06-11


### Filter to ONLY for 5 Stocks + Rename FB to META

In [13]:
# Rename FB to META (because stock data uses META.csv)
df_news.loc[df_news['stock'] == 'FB', 'stock'] = 'META'

# Define your 5 target stocks
target_stocks = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']

# Filter news to only these 5 stocks
df_news = df_news[df_news['stock'].isin(target_stocks)]

print(f"\nNews after filtering to 5 stocks: {len(df_news):,} articles")
print("\nCount per stock:")
print(df_news['stock'].value_counts())


News after filtering to 5 stocks: 5,444 articles

Count per stock:
stock
NVDA    3146
GOOG    1199
AAPL     441
META     380
AMZN     278
Name: count, dtype: int64


### Load stocks and collect trading days

In [17]:
tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']
all_trading_days = set()

for ticker in tickers:
    df = load_stock(ticker)
    df = fix_dtypes(df)
    df = handle_missing(df)
    
    # Add all trading days from this stock
    all_trading_days.update(df['Date'].dt.date)
    
    print(f"{ticker}: {len(df)} rows, {df['Date'].min().date()} to {df['Date'].max().date()}")

print(f"\nTotal unique trading days from all 5 stocks: {len(all_trading_days):,}")

AAPL: 3774 rows, 2009-01-02 to 2023-12-29
AMZN: 3774 rows, 2009-01-02 to 2023-12-29
GOOG: 3774 rows, 2009-01-02 to 2023-12-29
META: 2923 rows, 2012-05-18 to 2023-12-29
NVDA: 3774 rows, 2009-01-02 to 2023-12-29

Total unique trading days from all 5 stocks: 3,774


Get trading days

In [14]:
def get_next_trading_day(date_obj, trading_days):
    """
    If date is a weekend or holiday, find the next trading day.
    Example: Saturday → Monday, Holiday → next open day
    """
    date_obj = pd.to_datetime(date_obj).date()
    while date_obj not in trading_days:
        date_obj = date_obj + pd.Timedelta(days=1)
    return date_obj

# Test the function
test_date = pd.to_datetime('2020-03-07').date() 
print(f"Test: {test_date} → {get_next_trading_day(test_date, all_trading_days)}")

Test: 2020-03-07 → 2020-03-09


Apply Alignment to News

In [16]:
# Align: keep date if trading day, otherwise shift to next trading day
df_news['trading_day'] = df_news['news_date'].apply(
    lambda x: x if x in all_trading_days else get_next_trading_day(x, all_trading_days)
)
print(f"Original unique news dates: {df_news['news_date'].nunique():,}")
print(f"Unique trading days after alignment: {df_news['trading_day'].nunique():,}")

Original unique news dates: 1,318
Unique trading days after alignment: 1,222
